# Minimal Study Notebook: Train + Inference

A compact training loop and decoding flow using your existing code infra.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.transformer.transformer_lm import TransformerLM
from src.training.optimizer import AdamW
from src.training.loss import cross_entropy
from src.training_loop.data_loading import get_batch
from src.training_loop.checkpointing import save_checkpoint, load_checkpoint


In [ ]:
# -------- Config (minimal) --------
train_data = REPO_ROOT / "artifacts/tokenized/tinystories_train_tokens.bin"
val_data = REPO_ROOT / "artifacts/tokenized/tinystories_val_tokens.bin"
memmap_dtype = "uint16"
vocab_size = 4096

# Fallback to byte-level if tokenized bins are missing
if not train_data.exists() or not val_data.exists():
    train_data = REPO_ROOT / "data/TinyStoriesV2-GPT4-train.txt"
    val_data = REPO_ROOT / "data/TinyStoriesV2-GPT4-valid.txt"
    memmap_dtype = "uint8"
    vocab_size = 256
    print("Using byte-level fallback data")

context_length = 128
d_model = 320
num_heads = 8
d_ff = 1280
num_layers = 6
theta = 10000.0

batch_size = 16
learning_rate = 3e-4
weight_decay = 0.01
betas = (0.9, 0.95)
eps = 1e-8

max_iters = 300
log_interval = 25
eval_interval = 100
eval_batches = 5

checkpoint_path = REPO_ROOT / "artifacts/checkpoints/study_minimal.pt"
resume = False

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
seed = 1337


In [ ]:
# -------- Data + model --------
torch.manual_seed(seed)
np.random.seed(seed)

dtype = np.dtype(memmap_dtype)
train_tokens = np.memmap(train_data, mode="r", dtype=dtype)
val_tokens = np.memmap(val_data, mode="r", dtype=dtype)
print(f"train tokens: {len(train_tokens):,}")
print(f"val tokens:   {len(val_tokens):,}")

model = TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers,
    theta=theta,
    device=device,
).to(device)

optimizer = AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
    betas=betas,
    eps=eps,
)

start_iter = 0
if resume and checkpoint_path.exists():
    start_iter = load_checkpoint(str(checkpoint_path), model, optimizer)
    print(f"Resumed from {checkpoint_path} @ iter {start_iter}")


In [ ]:
@torch.no_grad()
def eval_loss(split_tokens, n_batches=5):
    model.eval()
    vals = []
    for _ in range(n_batches):
        x, y = get_batch(split_tokens, batch_size, context_length, device)
        logits = model(x)
        b, t, v = logits.shape
        loss = cross_entropy(logits.view(b * t, v), y.view(b * t))
        vals.append(loss.item())
    model.train()
    return float(np.mean(vals))


In [ ]:
# -------- Minimal training loop --------
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

model.train()
last_t = time.time()

for it in range(start_iter, max_iters):
    x, y = get_batch(train_tokens, batch_size, context_length, device)
    logits = model(x)
    b, t, v = logits.shape
    loss = cross_entropy(logits.view(b * t, v), y.view(b * t))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if (it + 1) % log_interval == 0:
        now = time.time()
        print(f"iter {it+1:4d} | train_loss {loss.item():.4f} | dt {now - last_t:.2f}s")
        last_t = now

    if (it + 1) % eval_interval == 0:
        tr = eval_loss(train_tokens, eval_batches)
        va = eval_loss(val_tokens, eval_batches)
        print(f"iter {it+1:4d} | train_eval {tr:.4f} | val_eval {va:.4f}")

save_checkpoint(model, optimizer, max_iters, str(checkpoint_path))
print(f"Saved checkpoint: {checkpoint_path}")


In [ ]:
# -------- Minimal inference (temperature sampling) --------
def encode_prompt(prompt: str):
    if memmap_dtype == "uint8":
        return list(prompt.encode("utf-8"))
    # For tokenized setup, this notebook stays minimal: expect prompt as space-separated token IDs.
    return [int(x) for x in prompt.strip().split()] if prompt.strip() else []


def decode_ids(ids):
    if memmap_dtype == "uint8":
        return bytes(ids).decode("utf-8", errors="replace")
    return " ".join(str(i) for i in ids)


@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120, temperature: float = 1.0):
    ids = encode_prompt(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        x_cond = x[:, -context_length:]
        logits = model(x_cond)
        next_logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return decode_ids(x[0].tolist())


# Byte-level example prompt. For tokenized mode, pass token IDs string like: "1 245 89"
sample = generate("Once upon a time", max_new_tokens=120, temperature=0.9)
print(sample)
